<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cleaning of Dataset 2

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
import warnings
import numpy as np

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df2 = pd.read_csv('Dataset 2.csv')

# Drops all unnecessary columns
df2 = df2.drop(columns=['age']) 

# Drops all with null values
df2.dropna(inplace=True)

# Remove Outliers
upper_quantile = df2['price'].quantile(0.99)
df2 = df2[df2['price'] <= upper_quantile]

# Log transforms price to reduce variablilty
df2['price'] = np.log(df2['price'])

# One-Hot encode all categorical features
categorical_cols_to_encode = df2.select_dtypes(include='object').columns
df2 = pd.get_dummies(df2, columns=categorical_cols_to_encode, drop_first=True)

# Find the target and numerical features
target_column = 'price'
numerical_features = ['area', 'bhk', 'bathroom']
y = df2[target_column].copy()

# Create scalar
scaler = StandardScaler()

# Create feature base without price
x_base = df2.drop(columns=[target_column]).copy()

x_linear = x_base.copy()

# Apply polynomial features to numerical columns for linear model
poly = PolynomialFeatures(degree=2, include_bias=False)
numerical_poly_features = poly.fit_transform(x_linear[numerical_features])
poly_feature_names = poly.get_feature_names_out(numerical_features)
x_linear_poly_df = pd.DataFrame(numerical_poly_features, columns=poly_feature_names, index=x_linear.index)

# Drop original numerical columns and concatenate polynomial features for linear model
x_linear = x_linear.drop(columns=numerical_features)
x_linear = pd.concat([x_linear, x_linear_poly_df], axis=1)

# Find all numerical features, even the created ones for linear model
all_numerical_features_linear = x_linear.select_dtypes(include=np.number).columns.tolist()

# Scale numerical features for linear model
x_linear[all_numerical_features_linear] = scaler.fit_transform(x_linear[all_numerical_features_linear])

x_tree = x_base.copy()

# Scale numerical features for tree models
x_tree[numerical_features] = scaler.fit_transform(x_tree[numerical_features])

# Splits the data 80/20 for training and testing the model respectfully
x_train_linear, x_test_linear, y_train_linear, y_test_linear = train_test_split(x_linear, y, test_size=0.2, random_state=42)
x_train_tree, x_test_tree, y_train_tree, y_test_tree = train_test_split(x_tree, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 2: Dataset 2.csv")
print("=" * 60)
print("\n--- Features for Linear Models (x_linear) ---")
print("\nFirst 5 rows:")
display(x_linear.head(5))
print("\nData types:")
print(x_linear.dtypes)
print("\nMissing values:")
print(x_linear.isnull().sum().sum())
print(f"Shape of x_linear: {x_linear.shape}")

print("\n--- Features for Tree Models (x_tree) ---")
print("\nFirst 5 rows:")
display(x_tree.head(5))
print("\nData types:")
print(x_tree.dtypes)
print("\nMissing values:")
print(x_tree.isnull().sum().sum())
print(f"Shape of x_tree: {x_tree.shape}")

print("\nTarget variable (price) statistics:")
print(y.describe())

# Price is in Lahk which is 100,000 Rupees (₹)

In [ ]:
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def rmse_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate RMSE on the original scale
    return np.sqrt(mean_squared_error(y_true_original, y_pred_original))

def mae_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate MAE on the original scale
    return mean_absolute_error(y_true_original, y_pred_original)

def r2_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate R2 on the original scale
    return r2_score(y_true_original, y_pred_original)

# Create scorers that can be used with GridSearchCV or cross_val_score
# 'neg_' prefix is used for metrics to be minimized (RMSE, MAE)
neg_rmse_original_scorer = make_scorer(rmse_original_scale, greater_is_better=False)
neg_mae_original_scorer = make_scorer(mae_original_scale, greater_is_better=False)
r2_original_scorer = make_scorer(r2_original_scale, greater_is_better=True)

### Linear Regression for Dataset 2

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

#Initializing Linear Regression Model and Running
lr = LinearRegression()
lr.fit(x_train_linear, y_train_linear)
y_pred = lr.predict(x_test_linear)

#Fetching the Performance Metrics For The Model
rmse = rmse_original_scale(y_test_linear, y_pred)
mae = mae_original_scale(y_test_linear, y_pred)
r2 = r2_original_scale(y_test_linear, y_pred)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score

#Graph Showing predicted vs Actual With Errors and Coefficents

#Printing Performance Metrics for the model
print(f"Root Mean Squared Error: {rmse:.2f}\n")
print(f"Mean Absolute Error: {mae:.2f}\n")
print(f"R^2 Score: {r2:.2f}\n")
print(f"\nCross-validation RMSE:\t₹{-cross_val_score(lr, x_linear, y, cv=5, scoring=neg_rmse_original_scorer).mean():,.2f}K")
print(f"Cross-validation MAE:\t₹{-cross_val_score(lr, x_linear, y, cv=5, scoring=neg_mae_original_scorer).mean():,.2f}K")
print(f"Cross-validation R²:\t{cross_val_score(lr, x_linear, y, cv=5, scoring=r2_original_scorer).mean():.4f}")

y_test_actual = np.exp(y_test_linear)
y_pred_actual = np.exp(y_pred)

#Constructing A Scatter Plot of Real Vs Expected Value to Vizualize the Perfomrance of the Model on The Test Set
plt.figure(figsize=(10, 6))
plt.scatter(y_test_actual, y_pred_actual, alpha=0.6, color='blue')
min_val = min(y_test_actual.min(), y_pred_actual.min())
max_val = max(y_test_actual.max(), y_pred_actual.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
plt.xlabel('Actual Prices (Lakh ₹)', fontsize=12)
plt.ylabel('Predicted Prices (Lakh ₹)', fontsize=12)
plt.title('Actual vs. Predicted Property Prices', fontsize=14)
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

In [ ]:
#Fetching Feature Names and Coefficents
feature_names = x_train_linear.columns
coefficients = lr.coef_

#Wrapping them together in a Pandas DataFrame for Easy Comparision and Viewing
feature_importance = pd.DataFrame({
 'Feature': feature_names,
 'Coefficient': coefficients
})

#Using the Absolute Function in the NumPy Library before Sorting the Coefficents in Decending Order
feature_importance['Abs_Coefficient'] = np.abs(feature_importance['Coefficient'])
top_10_features = feature_importance.sort_values(by='Abs_Coefficient', ascending=False).head(10)

#Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
 data=top_10_features, 
 x='Coefficient', 
 y='Feature', 
)
plt.title('Top 10 Features Driving Property Prices', fontsize=15)
plt.xlabel('Coefficient Value (Impact on Price in Lakh ₹)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features[['Feature', 'Coefficient']])

The Linear Regression model is able to predict the prices of cheap and standard homes very well, which accounts for its high R^2 score of 0.85. However, as the prices of the properties increase, the model begins to break down. The most important features seem to be the real estate companies themselves and their locations, as these were the features that made it into the top 10. This may indicate that for this specific dataset, "builder_Shree sakthivel realestate" and "builder_Vinay Asrani" may have made up a majority of the middle-to-high-end real estate, or perhaps they only had middle-to-high-end real estate within this sample.

The model then became dependent on these two features to make predictions on higher-cost housing, which may explain the low accuracy once the price of the houses crossed 200 Lakh. This suggests that a dataset with more high-end housing from a variety of real estate companies may be required. When the datasets are joined, we should expect the R^2 value to reduce, but the accuracy at the top of the price bracket should improve as the model finds more reliable predictors.This hypothesis is further backed up by a Mean Squared Error (MSE) that greatly exceeds the Mean Absolute Error (MAE). The MAE is fairly small and manageable since it doesn't penalize large deviations to the same extent as the MSE, which is orders of magnitude larger. Even when using the Root Mean Squared Error (RMSE), the value is still over double the MAE, further demonstrating the ineffectiveness of the model at the higher end of the dataset.

Therefore the core problems with the linear regression model for this dataset is the massive error at the top. In statistics you would remedy this with a log transformation. This "squishes" the data closer together and may allow the model to become less reliant on the real estate corporations themselves to predict the value of the house. Additionally a non linear model like RF or even a NN may help at the cost of increased computing cost. Additionally you could even go as far to sort the houses into groups(price ranges) and have the model focus on sorting the houses into these price ranges which should reduce the noise. 

### Random Forest Regressor on Dataset 2

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 238, 300], # Number of trees in the forest
    'max_features': ['sqrt', 'log2', 1.0, 0.5, 0.3], # Number of features to consider at each split
    'min_samples_leaf': [1, 2, 4], # Minimum number of samples required at a leaf node
    'max_depth': [10, 30, 54, 60] # Maximum depth of the tree
}

# Initialize GridSearchCV
# We use the negative root mean squared error as our scoring metric, 
# as GridSearchCV tries to maximize the score, so we negate RMSE.
grid_search = GridSearchCV(
    estimator = RandomForestRegressor(random_state=42), # Use a fixed random_state for reproducibility
    param_grid = param_grid,
    cv = 5, # 5-fold cross-validation
    scoring = r2_original_scorer, # Metric to optimize
    n_jobs = -1, # Use all available CPU cores
    verbose = 2 # Display progress
)

# Fit GridSearchCV to the training data
print("Starting GridSearchCV...")
grid_search.fit(x_train_tree, y_train_tree)
print("GridSearchCV completed.")

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = abs(grid_search.best_score_)

print(f"\nBest Parameters: {best_params}")
print(f"Best Cross-Validation R²: {best_score:,.4f}")

# You can also get the best estimator found by GridSearchCV
best_rfr_model = grid_search.best_estimator_

# Evaluate the best model on the test set
y_pred_tuned = best_rfr_model.predict(x_test_tree)
rmse_tuned = rmse_original_scale(y_test_tree, y_pred_tuned)
mae_tuned = mae_original_scale(y_test_tree, y_pred_tuned)
r2_tuned = r2_original_scale(y_test_tree, y_pred_tuned)

print("\n--- Tuned Random Forest Model Performance ---")
print(f"RMSE on Test Set:\t₹{rmse_tuned:,.2f}K")
print(f"MAE on Test Set:\t₹{mae_tuned:,.2f}K")
print(f"R² on Test Set:\t{r2_tuned:.4f}")

# You can also run cross-validation on the best model separately for comparison
cv_rmse_tuned = -cross_val_score(best_rfr_model, x_tree, y, cv=5, scoring=neg_rmse_original_scorer).mean()
cv_mae_tuned = -cross_val_score(best_rfr_model, x_tree, y, cv=5, scoring=neg_mae_original_scorer).mean()
cv_r2_tuned = cross_val_score(best_rfr_model, x_tree, y, cv=5, scoring=r2_original_scorer).mean()

print(f"\nCross-validation RMSE (Best Model):\t₹{cv_rmse_tuned:,.2f}K")
print(f"Cross-validation MAE (Best Model):\t₹{cv_mae_tuned:,.2f}K")
print(f"Cross-validation R² (Best Model):\t{cv_r2_tuned:.4f}")

y_test_original = np.exp(y_test_tree)
y_pred_original = np.exp(y_pred_tuned)

# Predicted vs Actual scatter
plt.figure(figsize=(10, 6))
plt.scatter(y_test_original, y_pred_original, alpha=0.5, s=16, color="blue")
plt.plot([0, y_test_original.max()], [0, y_test_original.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Prices (Lakh ₹)', fontsize=12)
plt.ylabel('Predicted Prices (Lakh ₹)', fontsize=12)
plt.title('Actual vs. Predicted Property Prices', fontsize=14)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Get feature importances from the Random Forest Regressor
feature_importances = best_rfr_model.feature_importances_
feature_names = x_tree.columns

# Create a DataFrame for feature importance
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

# Visualization: Feature Category Importance Breakdown
area_importance = feature_importance_df[feature_importance_df['Feature'] == 'area']['Importance'].values[0]
bhk_importance = feature_importance_df[feature_importance_df['Feature'] == 'bhk']['Importance'].values[0]
bathroom_importance = feature_importance_df[feature_importance_df['Feature'] == 'bathroom']['Importance'].values[0]

structural_features = area_importance + bhk_importance + bathroom_importance

location_features = feature_importance_df[feature_importance_df['Feature'].str.startswith('location_')]['Importance'].sum()
builder_features = feature_importance_df[feature_importance_df['Feature'].str.startswith('builder_')]['Importance'].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Pie chart of feature categories
categories = ['Structural\n(Area, BHK, Bathroom)', 'Location', 'Builder']
importances = [structural_features, location_features, builder_features]
colors = ['#2E86AB', '#A23B72', '#F18F01']

axes[0].pie(importances, labels=categories, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Feature Importance by Category', fontsize=14, fontweight='bold')

# Detailed breakdown of top features
top_10_features = feature_importance_df.sort_values(by='Importance', ascending=False).head(10)

axes[1].barh(range(len(top_10_features)), top_10_features['Importance'].values)
axes[1].set_yticks(range(len(top_10_features)))
axes[1].set_yticklabels(top_10_features['Feature'].values, fontsize=10)
axes[1].set_xlabel('Feature Importance', fontsize=12)
axes[1].set_title('Top 10 Features in Detail', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFeature Importance Breakdown:")
print(f"Structural Features: {structural_features*100:.2f}%")
print(f"Location Features: {location_features*100:.2f}%")
print(f"Builder Features: {builder_features*100:.2f}%")

In [ ]:
price_ranges = [
    (y_test_original.min(), y_test_original.quantile(0.33), "Low"),
    (y_test_original.quantile(0.33), y_test_original.quantile(0.67), "Medium"),
    (y_test_original.quantile(0.67), y_test_original.max(), "High")
]

for min_p, max_p, label in price_ranges:
    mask = (y_test_original >= min_p) & (y_test_original <= max_p)
    if mask.sum() > 0:
        range_rmse = np.sqrt(np.mean((y_test_original[mask] - y_pred_original[mask])**2))
        range_mae = np.mean(np.abs(y_test_original[mask] - y_pred_original[mask]))
        range_r2 = r2_score(y_test_original[mask], y_pred_original[mask])
        
        print(f"\n{label} Price Range (₹{min_p:.0f}K - ₹{max_p:.0f}K):")
        print(f"  - RMSE: ₹{range_rmse:.2f}K")
        print(f"  - MAE: ₹{range_mae:.2f}K")
        print(f"  - R²: {range_r2:.4f}")
        print(f"  - Sample size: {mask.sum()}")

### Analysis

The analysis of the Random Forests shows that the prediction of property prices is fundamentally basedon a mix of structural and spatial factors, although hierarchically the most significant ones are clear:

**1. Structural Dominance (74.6% of predictive power)**
- **Area (50.4%)**: Area is by far the most important price determinant. This feature alone serves half of all predictions and is based on the idea that the bigger the property, the higher the price.
- **BHK and Bathrooms (24.2%)**: These criteria have secondary yet significant impact. Nevertheless, they are very much correlated with area, bigger houses obviously have more rooms and that is why they become less significant when area is present.

**2. Location Effects (18.5% of predictive power)**
- Location features as a group make around a fifth of the model predictions, but crucially, location features help reveal non-linear spatial patterns that cannot be identified by simpler linear models.
- Certain neighborhoods (Veppampattu, West Mambalam, T Nagar) exhibit different price impacts, suggesting that location is a multiplicative, rather than an additive, factor.

**3. Builder Identity (6.6% of predictive power)**
- Although the reputation of a builder has little average effect, some developers ( Viswaraj, MEHTA REAL ESTATE) display some observable effect, probably due to brand value and reputation of quality of construction.

### Model Performance Strengths

The Random Forest achieves R² of 0.87 on the test set, indicating that the model explains 87% of price variance. Notably:
- **High-end performance**: The model performs well on predicting costly properties (R2 = 0.79 in high-price range) where linear models usually fail.
- **Minimal overfitting**: The cross-validation R 2 of 0.72 is near the test R 2 of 0.87, indicating that the model is not overfitting.

### Key Takeaways for Property Stakeholders

1. **For Buyers/Sellers**: Property size (area) should be given priority.
2. **For Investors**: There are opportunities for upselling in wealthier locations.
3. **For Developers**: Builder reputation, while less important than expected, does influence pricing and could be leveraged for premium positioning.


### Gradient Boosting on Dataset 2

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV
import numpy as np
import matplotlib.pyplot as plt

# Define the parameter grid to search for XGBoost
param_grid_xgb = {
    "colsample_bytree": [0, 0.3, 0.5],
    "gamma": [0, 0.3, 0.5],
    "learning_rate": [0.03, 0.1, 0.29, 0.3], # default 0.1 
    "max_depth": [2, 3, 4, 6], # default 3
    "n_estimators": [200, 400, 600, 700], # default 100
    "subsample": [1, 0.7, 0.5, 0.3, 0]
}

# Initialize GridSearchCV for XGBoost
print("Starting GridSearchCV for XGBoost...")
grid_search_xgb = GridSearchCV(
    estimator=XGBRegressor(objective="reg:squarederror", random_state=42),
    param_grid=param_grid_xgb,
    cv=5,
    scoring=r2_original_scorer, # Using the custom R2 scorer
    n_jobs=-1,
    verbose=2
)

# Fit GridSearchCV to the training data
grid_search_xgb.fit(x_train_tree, y_train_tree)
print("GridSearchCV for XGBoost completed.")

# Get the best parameters and best score
best_params_xgb = grid_search_xgb.best_params_
best_score_xgb = grid_search_xgb.best_score_

print(f"\nBest Parameters for XGBoost: {best_params_xgb}")
print(f"Best Cross-Validation R² for XGBoost: {best_score_xgb:.4f}")

# Get the best estimator found by GridSearchCV
best_xgb_model = grid_search_xgb.best_estimator_

# Evaluate the best XGBoost model on the test set
y_pred_xgb = best_xgb_model.predict(x_test_tree)

# Evaluate on original scale
rmse_xgb = rmse_original_scale(y_test_tree, y_pred_xgb)
mae_xgb = mae_original_scale(y_test_tree, y_pred_xgb)
r2_xgb = r2_original_scale(y_test_tree, y_pred_xgb)

print("\n--- Tuned XGBoost Regressor Performance ---")
print(f"RMSE on Test Set:\t₹{rmse_xgb:,.2f}K")
print(f"MAE on Test Set:\t₹{mae_xgb:,.2f}K")
print(f"R² on Test Set:\t{r2_xgb:.4f}")

# Cross-validation for the best model
cv_rmse_xgb = -cross_val_score(best_xgb_model, x_tree, y, cv=5, scoring=neg_rmse_original_scorer).mean()
cv_mae_xgb = -cross_val_score(best_xgb_model, x_tree, y, cv=5, scoring=neg_mae_original_scorer).mean()
cv_r2_xgb = cross_val_score(best_xgb_model, x_tree, y, cv=5, scoring=r2_original_scorer).mean()

print(f"\nCross-validation RMSE (Best XGBoost Model):\t₹{cv_rmse_xgb:,.2f}K")
print(f"Cross-validation MAE (Best XGBoost Model):\t₹{cv_mae_xgb:,.2f}K")
print(f"Cross-validation R² (Best XGBoost Model):\t{cv_r2_xgb:.4f}")

y_test_original = np.exp(y_test_tree)
y_pred_original = np.exp(y_pred_xgb)

# Predicted vs Actual scatter
plt.figure(figsize=(8, 6))
plt.scatter(y_test_original, y_pred_original, alpha=0.4, s=15)
plt.plot([0, y_test_original.max()], [0, y_test_original.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price (Lakh ₹)')
plt.ylabel('Predicted Price (Lakh ₹)')
plt.title('XGBoost Regressor: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

# Fetching Feature Names and Importances from XGBoost
feature_names = x_train_tree.columns
importances = best_xgb_model.feature_importances_

# Wrapping them together in a Pandas DataFrame for Easy Comparison and Viewing
feature_importance_xgb = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sorting Feature Importances in Descending Order
top_10_features_xgb = feature_importance_xgb.sort_values(by='Importance', ascending=False).head(10)

# Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_10_features_xgb,
    x='Importance',
    y='Feature',
)
plt.title('Top 10 Features Driving Property Prices (XGBoost)', fontsize=15)
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features_xgb[['Feature', 'Importance']])